# Train

In [ ]:
import resource
try:
    soft, hard = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (min(500000, hard), hard))
    print(f"File descriptor limit increased to {min(500000, hard)}")
except Exception as e:
    print(f"Could not increase limit: {e}")

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"]="3"

In [ ]:
import torch
import lightning.pytorch as pl
from omegaconf import OmegaConf
import glob

import nemo
import nemo.collections.asr as nemo_asr
from nemo.utils.exp_manager import exp_manager

from my_help_functions.norms import (
    BiasWeightChangeBatchNormNothing,
    BiasWeightChangeBatchNormOnlyMean,
    BiasWeightChangeBatchNormOnlyBias,
    BiasWeightChangeBatchNorm
)
from my_help_functions.norms import replace_batchnorm_with_norm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
data_dir = './sound_dataset_demo/'
MODEL_CONFIG = "matchboxnet_3x2x64_v1.yaml"
dataset_path = 'google_speech_recognition_v1'
dataset_basedir = os.path.join(data_dir, dataset_path)
config_path = f"configs/{MODEL_CONFIG}"
config = OmegaConf.load(config_path)
config = OmegaConf.to_container(config, resolve=True)
config = OmegaConf.create(config)
config.model.train_ds.manifest_filepath = os.path.join(dataset_basedir, 'train_manifest.json')
config.model.validation_ds.manifest_filepath = os.path.join(dataset_basedir, 'validation_manifest.json')
config.model.test_ds.manifest_filepath = os.path.join(dataset_basedir, 'validation_manifest.json')

In [ ]:
accelerator = 'gpu'
config.trainer.devices = 1
config.trainer.num_nodes = 1
config.trainer.accelerator = accelerator
config.trainer.strategy = 'auto'

print("Trainer config - \n")
print(OmegaConf.to_yaml(config.trainer))

trainer = pl.Trainer(precision=16, enable_progress_bar=False, **config.trainer)

In [ ]:
exp_dir = exp_manager(trainer, config.get("exp_manager", None))
exp_dir = str(exp_dir)
exp_dir

In [ ]:
asr_model = nemo_asr.models.EncDecClassificationModel(cfg=config.model, trainer=trainer)

# choose any
norm = BiasWeightChangeBatchNormNothing
#         BiasWeightChangeBatchNormOnlyMean,
#         BiasWeightChangeBatchNormOnlyBias,
#         BiasWeightChangeBatchNorm

asr_model = replace_batchnorm_with_norm(asr_model, norm, device)

In [ ]:
trainer.fit(asr_model)

# Test


In [ ]:
import torch
import json
import nemo.collections.asr as nemo_asr
import re

In [ ]:
#####
# Run, if you need to upload your model from your checkpoint
# Also, if needed, run some cells in Training section (i.e. with dataset_basedir or with config)
#####
'''
def best_ckpt_index(paths):
    best_idx = 0
    best_loss = float("inf")

    pattern = re.compile(r"val_loss=([0-9.]+)")

    for i, p in enumerate(paths):
        m = pattern.search(p)
        if not m:
            continue
        loss = float(m.group(1))
        if loss < best_loss:
            best_loss = loss
            best_idx = i

    return best_idx

# exp dir - insert your dir from training section
checkpoint_dir = os.path.join(exp_dir, 'checkpoints')
checkpoint_paths = list(glob.glob(os.path.join(checkpoint_dir, "*.ckpt")))
best_checkpoint = checkpoint_paths[best_ckpt_index(checkpoint_paths)]
checkpoint = torch.load(best_checkpoint)

asr_model = nemo_asr.models.EncDecClassificationModel(cfg=config.model, trainer=trainer)
asr_model = replace_batchnorm_with_norm(asr_model, norm, device)
asr_model.load_state_dict(checkpoint['state_dict'])
''';

In [ ]:
asr_model.setup_test_data(
    test_data_config={
        'manifest_filepath': os.path.join(dataset_basedir, 'test_manifest.json'),
        'sample_rate': 16000,
        'labels': asr_model.cfg.labels,
        'batch_size': 4096,
        'shuffle': False,
        'num_workers': 4
    }
)
trainer.test(asr_model, ckpt_path=None)